# 03 - Evaluation, Explainability, and Error Analysis

This notebook analyzes the selected validation model, explains global and local TF-IDF coefficient behavior, and records the final one-time test evaluation in the final section. Do not tune models after viewing test results.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import (
    build_prediction_analysis_frame,
    classification_report_dataframe,
    summarize_class_errors,
    summarize_confusion_pairs,
    validation_summary,
)
from src.paths import MODELS_DIR, PROCESSED_DATA_DIR, REPORTS_DIR
from src.preprocessing import preprocess_text

pd.set_option("display.max_colwidth", 140)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## Load Validation Split and Selected Validation Model

The validation model is trained on the train split only. It is used for error analysis and explanation before the final train+validation model is evaluated once on the test split.

In [ ]:
DATASET_NAME = "QCRI/HumAID-all"


def load_split(split: str) -> pd.DataFrame:
    local_path = PROCESSED_DATA_DIR / f"humaid_{split}_minimal.parquet"
    if local_path.exists():
        return pd.read_parquet(local_path)

    dataset = load_dataset(DATASET_NAME, split=split)
    frame = dataset.to_pandas()
    frame["text_minimal"] = frame["tweet_text"].apply(preprocess_text)
    return frame


train_df = load_split("train")
validation_df = load_split("validation")
X_val_raw = validation_df["tweet_text"]
y_val = validation_df["class_label"]

validation_model_path = MODELS_DIR / "selected_validation_model_e11.joblib"
if not validation_model_path.exists():
    raise FileNotFoundError(
        f"Missing {validation_model_path}. Run notebooks/02_model_training.ipynb first."
    )

validation_model = joblib.load(validation_model_path)
classifier = validation_model.named_steps["classifier"]
vectorizer = validation_model.named_steps["vectorizer"]
class_names = list(classifier.classes_)

print("Validation samples:", len(validation_df))
print("Classes:", len(class_names))
print("Features:", len(vectorizer.get_feature_names_out()))

## Validation Prediction Analysis

In [ ]:
validation_predictions = validation_model.predict(X_val_raw)
validation_probabilities = validation_model.predict_proba(X_val_raw)

validation_analysis_df = build_prediction_analysis_frame(
    texts=X_val_raw,
    y_true=y_val,
    y_pred=validation_predictions,
    probabilities=validation_probabilities,
)
summary = validation_summary(validation_analysis_df, y_val, validation_predictions)
summary

In [ ]:
wrong_predictions_df = (
    validation_analysis_df.loc[~validation_analysis_df["is_correct"]]
    .sort_values(["confidence", "confidence_margin"], ascending=False)
    .reset_index(drop=True)
)

wrong_predictions_df[[
    "text",
    "true_label",
    "predicted_label",
    "confidence",
    "confidence_margin",
]].head(20)

In [ ]:
validation_confusion_pairs_df = summarize_confusion_pairs(validation_analysis_df)
validation_report_df = classification_report_dataframe(y_val, validation_predictions, labels=class_names)
class_error_summary_df = summarize_class_errors(validation_analysis_df)

validation_analysis_df.to_csv(REPORTS_DIR / "validation_prediction_analysis.csv", index=False)
wrong_predictions_df.to_csv(REPORTS_DIR / "validation_wrong_predictions.csv", index=False)
validation_confusion_pairs_df.to_csv(REPORTS_DIR / "validation_confusion_pairs.csv", index=False)
validation_report_df.to_csv(REPORTS_DIR / "validation_classification_report.csv")

validation_confusion_pairs_df.head(20), class_error_summary_df.head(10)

## Other Relevant Information Ambiguity

`other_relevant_information` is broad and often absorbs crisis-related updates that do not cleanly match operational categories. These rows should be interpreted as a mix of model error, ambiguous labels, and likely annotation noise.

In [ ]:
other_errors_df = wrong_predictions_df.loc[wrong_predictions_df["true_label"] == "other_relevant_information"]
other_error_distribution_df = (
    other_errors_df["predicted_label"]
    .value_counts()
    .rename_axis("predicted_label")
    .reset_index(name="error_count")
)
other_error_distribution_df["error_percentage"] = (
    other_error_distribution_df["error_count"] / max(len(other_errors_df), 1) * 100
)
other_error_distribution_df

In [ ]:
high_confidence_thresholds = [0.60, 0.70, 0.80, 0.90]
high_confidence_summary_df = pd.DataFrame(
    {
        "threshold": threshold,
        "wrong_predictions": int((wrong_predictions_df["confidence"] >= threshold).sum()),
        "share_of_wrong_predictions": float((wrong_predictions_df["confidence"] >= threshold).mean()),
    }
    for threshold in high_confidence_thresholds
)
high_confidence_summary_df

## Manual Label Audit

The notebook can create a local manual-audit worksheet for high-confidence errors. The detailed worksheet is ignored by Git because it contains full message text; only aggregate audit summaries are safe to publish.

In [ ]:
manual_audit_path = REPORTS_DIR / "manual_error_audit.csv"
manual_summary_path = REPORTS_DIR / "manual_error_audit_summary.csv"

if manual_audit_path.exists():
    manual_audit_df = pd.read_csv(manual_audit_path)
else:
    audit_targets = [
        "other_relevant_information",
        "requests_or_urgent_needs",
        "missing_or_found_people",
    ]
    manual_audit_df = (
        wrong_predictions_df.loc[wrong_predictions_df["true_label"].isin(audit_targets)]
        .sort_values(["confidence", "confidence_margin"], ascending=False)
        .head(30)
        .copy()
    )
    manual_audit_df["audit_decision"] = ""
    manual_audit_df["audit_notes"] = ""
    manual_audit_df.to_csv(manual_audit_path, index=False)

completed_audit_df = manual_audit_df.loc[manual_audit_df["audit_decision"].astype(str).str.strip().ne("")]
if not completed_audit_df.empty:
    manual_audit_summary_df = (
        completed_audit_df["audit_decision"]
        .value_counts()
        .rename_axis("audit_decision")
        .reset_index(name="sample_count")
    )
    manual_audit_summary_df["percentage"] = (
        manual_audit_summary_df["sample_count"] / len(completed_audit_df) * 100
    )
    manual_audit_summary_df.to_csv(manual_summary_path, index=False)
elif manual_summary_path.exists():
    manual_audit_summary_df = pd.read_csv(manual_summary_path)
else:
    manual_audit_summary_df = pd.DataFrame(columns=["audit_decision", "sample_count", "percentage"])

manual_audit_summary_df

## Global Feature Importance

For linear Logistic Regression, feature weights show evidence the model associates with each class. They are not causal explanations and can include event-specific shortcuts.

In [ ]:
def extract_all_top_features(pipeline, top_n: int = 15) -> pd.DataFrame:
    vectorizer = pipeline.named_steps["vectorizer"]
    classifier = pipeline.named_steps["classifier"]
    feature_names = vectorizer.get_feature_names_out()
    records = []

    for class_index, class_name in enumerate(classifier.classes_):
        class_weights = classifier.coef_[class_index]
        positive_indices = np.argsort(class_weights)[-top_n:][::-1]
        negative_indices = np.argsort(class_weights)[:top_n]

        for rank, feature_index in enumerate(positive_indices, start=1):
            records.append(
                {
                    "class_name": class_name,
                    "direction": "positive",
                    "rank": rank,
                    "feature": feature_names[feature_index],
                    "weight": float(class_weights[feature_index]),
                }
            )

        for rank, feature_index in enumerate(negative_indices, start=1):
            records.append(
                {
                    "class_name": class_name,
                    "direction": "negative",
                    "rank": rank,
                    "feature": feature_names[feature_index],
                    "weight": float(class_weights[feature_index]),
                }
            )

    return pd.DataFrame(records)


top_features_df = extract_all_top_features(validation_model, top_n=15)
top_features_df.to_csv(REPORTS_DIR / "top_features_all_classes.csv", index=False)
top_features_df.head(30)

## Local Feature Contributions

For one message and one class, each local contribution is `TF-IDF value x class coefficient`. These contributions explain the linear score, not the final class probability.

In [ ]:
def explain_prediction(text: str, pipeline, top_n: int = 10) -> dict[str, object]:
    vectorizer = pipeline.named_steps["vectorizer"]
    classifier = pipeline.named_steps["classifier"]
    feature_names = vectorizer.get_feature_names_out()

    probabilities = pipeline.predict_proba([text])[0]
    predicted_index = int(np.argmax(probabilities))
    predicted_class = str(classifier.classes_[predicted_index])

    text_vector = vectorizer.transform([text])
    active_indices = text_vector.indices
    active_values = text_vector.data
    class_weights = classifier.coef_[predicted_index]
    contributions = active_values * class_weights[active_indices]

    contribution_df = pd.DataFrame(
        {
            "feature": feature_names[active_indices],
            "tfidf_value": active_values,
            "class_weight": class_weights[active_indices],
            "contribution": contributions,
        }
    ).sort_values("contribution", ascending=False)

    return {
        "predicted_class": predicted_class,
        "confidence": float(probabilities[predicted_index]),
        "positive_contributions": contribution_df.loc[contribution_df["contribution"] > 0].head(top_n),
        "negative_contributions": contribution_df.loc[contribution_df["contribution"] < 0].tail(top_n),
    }


sample_message = "Families urgently need clean water, food and medical supplies."
local_explanation = explain_prediction(sample_message, validation_model, top_n=10)
local_explanation["positive_contributions"].to_csv(REPORTS_DIR / "sample_local_explanation.csv", index=False)
local_explanation

## Spurious and Event-Specific Correlation Audit

Terms such as `Maryland`, `California`, `guardsman`, and event-specific wording can become shortcuts when particular events dominate a label. These checks are prompts for human review, not automatic proof of bias.

In [ ]:
event_terms = ["maryland", "california", "guardsman", "ecuador", "hurricane", "wildfire"]
event_term_records = []

for term in event_terms:
    mask = train_df["tweet_text"].str.contains(term, case=False, regex=False, na=False)
    counts = train_df.loc[mask, "class_label"].value_counts()
    for class_label, count in counts.items():
        event_term_records.append({"term": term, "class_label": class_label, "train_count": int(count)})

event_term_audit_df = pd.DataFrame(event_term_records)
event_term_audit_df.sort_values(["term", "train_count"], ascending=[True, False])

## Validation Analysis Takeaways

Common error modes include broad `other_relevant_information` ambiguity, urgent-needs versus donation confusion, missing-person versus injured/dead wording overlap, and event-specific lexical shortcuts. Interpret these as decision-support signals and keep a human reviewer in the loop.